<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex01-python-fundamentals/Ex01_03_plotting_light.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the set's library files from the public course repository, so nothing needs uploading.*
*This is the **light** version: every TODO is written out, and you only replace the `...` marked lines with what the comment beside them says.*


<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference text — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Prince, *Understanding Deep Learning*, MIT Press 2023.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# Ex_1 · Notebook 03 — figures an engineer can defend

matplotlib for engineering work: the figure and axes objects, labels with
units, several curves on one pair of axes, log scales, residual plots,
subplots, and saving at a resolution somebody can read.

## Why a whole notebook on plotting

Because from Ex_7 onwards, the figure **is** the result. A physics-informed
network produces no table of validated numbers; it produces a field, and what
you can say about it you will say with a picture — the solution against a
reference, the residual across the domain, the loss against epoch. If the
picture is unreadable, the work is unreviewable.

There is also a narrower reason. Every plotting mistake in this course is one
of four:

1. an axis with no label, or a label with no unit;
2. a linear axis showing a quantity that spans four decades;
3. two curves that cannot be told apart, because nothing distinguishes them
   beyond colour;
4. a figure assembled across several cells, which therefore never appears.

All four are avoidable in the same twenty minutes.

## The one rule

**Every axis is labelled, and every label carries its unit.** A plot of
deflection against position with bare axes is not a result; it is a shape. The
`engineering_axes` helper in `Ex_1_core.py` applies this course's conventions —
labels, a light grid, no box — and it takes the labels from you, with the units
already in them, because no helper can invent them.

In [ ]:
# files-cell v1 ----------------------------------------------------------
# This set's library files must sit beside the notebook. Locally they
# already do. On Google Colab, where a notebook opens on its own, they are
# fetched from the public course repository. Run this cell first.
import os, urllib.request
FILES = ['Ex_1_core.py']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex01-python-fundamentals/"
for f in FILES:
    if not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
print("files ready:", ", ".join(FILES))


In [ ]:
# --- setup -------------------------------------------------------------
# Needs Ex_1_core.py alongside this notebook.
import os
for f in ("Ex_1_core.py",):
    assert os.path.exists(f), f"{f} is missing - run the files cell above first"

from Ex_1_core import *                             # noqa: F401,F403
import numpy as np
import matplotlib.pyplot as plt

set_seed(88)
np.set_printoptions(precision=4, suppress=True)
print("setup complete")

## 1 · Figure and axes

matplotlib has two interfaces and mixing them is the source of most confusion
found online.

The **state-machine** interface — `plt.plot(...)`, `plt.xlabel(...)` — draws
into whichever figure is currently active. It is fine for one throwaway plot
and it becomes unmanageable the moment you have two.

The **object-oriented** interface, which this course uses everywhere, gives you
the objects explicitly:

```python
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(x, y)
ax.set_xlabel("position [m]")
plt.show()
```

`fig` is the whole canvas — its size, its title, and what gets saved. `ax` is
one set of axes inside it — its data, its labels, its limits, its legend. When
you have four panels you have one `fig` and four `ax` objects, and every call
says which panel it means. There is no ambiguity to be confused by.

`figsize` is in **inches**, and it matters more than it sounds: it sets the size
of the text relative to the data. A figure drawn at `(12, 8)` and then shrunk
into a report has unreadable labels. Draw it at the size it will be printed,
which for a single column is roughly 6 by 3.5.

**Keep one figure in one cell.** Jupyter renders a figure when the cell that
created it finishes. Build it across two cells and the second half is drawn
into a figure that has already been displayed, so it silently disappears. End
each plotting cell with `plt.show()`.

In [ ]:
x = np.linspace(0.0, 2.0, 200)               # position along the beam, m
y = beam_deflection(x)                       # deflection, mm

fig, ax = plt.subplots(figsize=(6.0, 3.5))
ax.plot(x, y, linewidth=2.0, color="#1f4e79")
engineering_axes(ax, "position from fixed end x [m]", "deflection [mm]",
                 title="Cantilever, 2 m, 500 N/m, S355")
ax.invert_yaxis()
plt.show()

print("tip deflection:", f"{y[-1]:.2f} mm  ({y[-1] / (2000) * 100:.3f} % of span)")

**What you should see.** A single curve rising from zero at the fixed end to
about 4 mm at the tip, drawn downwards, with both axes labelled and a title.
Then the printed tip deflection and its ratio to the span — about 0.2 %, which
is comfortably inside the usual serviceability limit of span/250, and which is
the kind of sentence a figure should let a reader write.

`ax.invert_yaxis()` is worth a moment. The deflection is downwards, so drawing
it downwards makes the picture agree with the beam. A figure that contradicts
the physical arrangement will be misread by somebody eventually.

## 2 · Several curves, and telling them apart

The commonest engineering figure is a family of curves: a parameter sweep, a
set of load cases, a comparison of models. Three rules make it readable.

**Distinguish by more than colour.** Some of your readers print in greyscale,
about one man in twelve has a colour vision deficiency, and projectors are
unreliable. Vary the line style or the marker as well as the colour. It costs
one keyword.

**Label the curves, not the legend.** `label="w = 250 N/m"` on each `plot` call,
then one `ax.legend()`. A legend entry reading `series 1` has told the reader
nothing.

**Order the legend the way the curves are ordered on the page**, when they have
a natural order. If the sweep goes from light to heavy, the legend should too;
matplotlib lists entries in the order you plotted them, so plot them in order.

### Your turn: a load sweep

Plot the deflection of the same cantilever for four distributed loads: 250,
500, 750 and 1000 N/m. Put all four on one pair of axes.

Requirements, all of which the check cell inspects:

- one curve per load, plotted in increasing order of load;
- each curve labelled with its load **and its unit**, in the form
  `"w = 250 N/m"`;
- a legend;
- both axes labelled with units, via `engineering_axes`;
- the y-axis inverted, because the beam deflects downwards;
- a different line style for each curve, not colour alone.

`beam_deflection(x, load=w)` gives you the curve for load `w`. Loop over the
four loads; a loop with four iterations that each draw a line is exactly what a
loop is for, and nothing here needs vectorising.

Line styles: `"-"`, `"--"`, `"-."`, `":"`. Cycle through them with `zip` or by
indexing.

In [ ]:
loads = [250.0, 500.0, 750.0, 1000.0]          # N/m
styles = ["-", "--", "-.", ":"]

fig, ax = plt.subplots(figsize=(6.5, 4.0))

# TODO 1 --- a load sweep, four labelled curves --------------------------------------
# Four `...` to replace, one per line:
#   line 1  ->  beam_deflection(x, load=w)          the curve for this load
#   line 2  ->  f"w = {w:.0f} N/m"                  the label, with its unit
#   line 3  ->  "deflection [mm]"                   the y label for engineering_axes
#   line 4  ->  ax.invert_yaxis()                   the beam deflects downwards
x = np.linspace(0.0, 2.0, 200)
for w, style in zip(loads, styles):
    y = ...                                       # <- beam_deflection(x, load=w)
    ax.plot(x, y, style, label=...)               # <- f"w = {w:.0f} N/m"
engineering_axes(ax, "position from fixed end x [m]", ..., legend=True)   # <- "deflection [mm]"
...                                               # <- ax.invert_yaxis()
plt.show()
# ------------------------------------------------------------------------------

In [ ]:
lines = ax.get_lines()
print(f"  {len(lines)} curves drawn")
labels = [ln.get_label() for ln in lines]
print("  labels:", labels)
print("  styles:", [ln.get_linestyle() for ln in lines])
print()
print("  x label:", repr(ax.get_xlabel()))
print("  y label:", repr(ax.get_ylabel()))
print("  legend present:", ax.get_legend() is not None)
print("  y axis inverted:", ax.get_ylim()[0] > ax.get_ylim()[1])

ok = (len(lines) == 4
      and all("N/m" in l for l in labels)
      and len(set(ln.get_linestyle() for ln in lines)) == 4
      and "[m]" in ax.get_xlabel() and "[mm]" in ax.get_ylabel()
      and ax.get_legend() is not None
      and ax.get_ylim()[0] > ax.get_ylim()[1])
print()
print("  PASS" if ok else "  FAIL", "- all six requirements")

**What you should see.** Four curves fanning out from the fixed end, the
heaviest load deflecting most, with a legend in load order; then `4 curves
drawn`, four distinct line styles, both labels containing units, and `PASS`.

The check cell inspects the axes object rather than the picture, which is worth
noticing in its own right: a matplotlib figure is a live object you can
interrogate, not an image. That is how you would write a test for a plotting
routine.

If the check reports zero curves, your plotting code and the check are in
different cells and the figure was already closed — put your drawing code in
the cell with the `fig, ax = plt.subplots(...)` line, as the template does.

## 3 · Log scales, and when a linear axis lies

A linear axis shows detail proportional to magnitude. If your quantity spans
four decades — and a training loss, a residual, and a convergence error all do
— then a linear axis shows you the first decade and compresses the rest onto
the baseline. You will stare at a flat line and conclude nothing is happening
while the quantity falls by a factor of a thousand.

Use `ax.set_yscale("log")` whenever the interesting behaviour is a *ratio*
rather than a difference. On a log axis, an exponential decay is a straight
line, and the slope is the rate — which means you can read a convergence rate
off the page instead of computing it.

From Ex_2 onwards every loss curve in this course is plotted on a log y-axis,
without exception. This cell is where that convention comes from.

The example below is the convergence of a numerical scheme: error against grid
spacing, for a first-order and a second-order method. On linear axes the two
are indistinguishable near zero. On log-log axes they are two straight lines of
slope 1 and 2, and the difference between the methods is the thing you see
first.

In [ ]:
h = np.logspace(-3, -1, 30)                  # grid spacing
err_first = 0.5 * h
err_second = 0.5 * h ** 2

fig, axes = plt.subplots(1, 2, figsize=(10.0, 3.8))

axes[0].plot(h, err_first, "-", label="first order")
axes[0].plot(h, err_second, "--", label="second order")
engineering_axes(axes[0], "grid spacing h [m]", "error [K]",
                 title="linear axes", legend=True)

axes[1].plot(h, err_first, "-", label="first order")
axes[1].plot(h, err_second, "--", label="second order")
axes[1].set_xscale("log")
axes[1].set_yscale("log")
engineering_axes(axes[1], "grid spacing h [m]", "error [K]",
                 title="log-log axes", legend=True)

fig.tight_layout()
plt.show()

**What you should see.** Two panels. On the left the two curves are almost on
top of each other and both look like "small"; on the right they are straight
lines of visibly different slope, and you can measure those slopes with a ruler
— rise over run gives 1 and 2, which are the orders of the two methods.

`fig.tight_layout()` stops the labels of one panel overlapping the next. Call
it on any multi-panel figure; there is no reason not to.

## 4 · The residual plot

A fit that looks good is not a fit that is good. The eye is poor at judging
small differences between two nearly-coincident curves, and excellent at
judging whether a cloud of points has structure. So plot the difference.

The convention, which you will use in every exercise from here on:

- top panel: the data and the model on the same axes, with the model as a line
  and the data as markers;
- bottom panel: the residual, data minus model, with a horizontal line at zero;
- shared x-axis, so a feature at one position lines up between the panels.

What you are looking for in the lower panel is **structure**. Residuals that
scatter evenly about zero mean the model has captured everything systematic and
what is left is noise. Residuals that curve, or that fan out, or that are all
positive at one end, mean the model is wrong in a way the upper panel hid from
you.

`plt.subplots(2, 1, sharex=True, gridspec_kw={"height_ratios": [3, 1]})` gives
the standard arrangement: two stacked panels, a shared x-axis, and the residual
panel a third the height of the data panel.

### Your turn

Build that figure for the beam. The measurements are `y_meas` at positions
`x_meas`; the model is `beam_deflection`. Requirements:

- upper panel: measurements as markers (`"o"`, `markersize=4`), model as a
  line, both labelled, a legend, y-axis labelled `"deflection [mm]"`;
- lower panel: the residual as markers, a horizontal line at zero
  (`axhline(0.0, ...)`), y-axis labelled `"residual [mm]"`, x-axis labelled
  `"position from fixed end x [m]"`;
- `engineering_axes` on both;
- `fig.tight_layout()` and `plt.show()`.

Mind the shapes. `y_meas` is `(25,)` and `beam_deflection(x_meas)` is `(25,)`,
so the subtraction is safe — but print both shapes anyway. That habit is the
whole of notebook 02, and it does not stop being useful because the topic has
changed.

In [ ]:
x_meas = np.linspace(0.0, 2.0, 25)
y_meas = measured_deflection(x_meas, noise_mm=0.06, seed=88)
y_model = beam_deflection(x_meas)

print("shapes:", x_meas.shape, y_meas.shape, y_model.shape)

fig, (ax_top, ax_bot) = plt.subplots(
    2, 1, figsize=(6.5, 5.0), sharex=True,
    gridspec_kw={"height_ratios": [3, 1]})

# TODO 2 --- data, model and residual ------------------------------------------------
# Four `...` to replace, one per line:
#   line 1  ->  y_meas                the measurements, as markers
#   line 2  ->  y_model               the model, as a line
#   line 3  ->  y_meas - y_model      the residual, data minus model
#   line 4  ->  0.0                   the horizontal line at zero
ax_top.plot(x_meas, ..., "o", markersize=4, label="measured")        # <- y_meas
ax_top.plot(x_meas, ..., "-", label="model")                         # <- y_model
engineering_axes(ax_top, "", "deflection [mm]", legend=True)

resid = ...                                                          # <- y_meas - y_model
ax_bot.plot(x_meas, resid, "o", markersize=4)
ax_bot.axhline(..., color="k", linewidth=0.8)                        # <- 0.0
engineering_axes(ax_bot, "position from fixed end x [m]", "residual [mm]")

fig.tight_layout()
plt.show()
# ------------------------------------------------------------------------------

In [ ]:
resid = y_meas - y_model
print(f"  residual mean {resid.mean():+.4f} mm, RMS {np.sqrt((resid ** 2).mean()):.4f} mm")
print(f"  the noise added was 0.0600 mm")
print()
print("  upper panel curves:", len(ax_top.get_lines()))
print("  lower panel lines :", len(ax_bot.get_lines()), "(markers plus the zero line)")
print("  y labels:", repr(ax_top.get_ylabel()), repr(ax_bot.get_ylabel()))
print("  x label :", repr(ax_bot.get_xlabel()))

ok = (len(ax_top.get_lines()) >= 2 and len(ax_bot.get_lines()) >= 2
      and "mm" in ax_top.get_ylabel() and "mm" in ax_bot.get_ylabel()
      and "m]" in ax_bot.get_xlabel())
print()
print("  PASS" if ok else "  FAIL", "- both panels populated and labelled")

**What you should see.** A two-panel figure; measurements sitting on the model
curve in the upper panel; and in the lower panel a band of points scattered
about zero with no visible trend, with an RMS close to the 0.06 mm of noise
that was added.

That absence of trend is the result. It says the analytic deflection formula
describes these measurements to within the instrument, and there is nothing
systematic left to explain. If the lower panel had curved — say, all positive
near the tip — you would be looking at evidence that the beam is not behaving
as a simple cantilever, and *that* is a finding worth reporting.

## 5 · Saving a figure

Screenshots of notebooks do not belong in reports. Save the figure.

```python
fig.savefig("cantilever.png", dpi=200, bbox_inches="tight")
```

- **`dpi=200`** or higher. The default of 100 is a screen resolution and looks
  soft on paper.
- **`bbox_inches="tight"`** trims the surrounding whitespace, which otherwise
  makes the figure look small when placed in a document.
- **PDF or SVG for anything with text in it**, if the document format allows.
  These are vector formats: the curve stays sharp at any zoom and the labels
  are real text rather than pixels. `fig.savefig("cantilever.pdf")` is all it
  takes.

On Colab, a saved file lands in the session's filesystem and vanishes when the
runtime restarts. Download it:

```python
from google.colab import files
files.download("cantilever.png")
```

The cell below saves the last figure both ways and reports the file sizes.

In [ ]:
fig.savefig("Ex01_residual.png", dpi=200, bbox_inches="tight")
fig.savefig("Ex01_residual.pdf", bbox_inches="tight")

for name in ("Ex01_residual.png", "Ex01_residual.pdf"):
    print(f"  {name:<22}{os.path.getsize(name) / 1024:8.1f} kB")

# On Colab, uncomment to download:
# from google.colab import files
# files.download("Ex01_residual.png")

**What you should see.** Two files, the PNG typically a few hundred kilobytes
and the PDF rather smaller, because a vector file stores the curve rather than
the pixels showing it.

## 6 · A checklist for every figure you hand in

Run through this before a figure leaves your notebook. It takes fifteen seconds
and it is the difference between a figure that gets discussed and one that gets
queried.

| | |
|---|---|
| Both axes labelled | including the quantity, not just the symbol |
| Units on both labels | in brackets: `deflection [mm]` |
| Log scale where the quantity spans decades | always for a loss curve |
| Curves distinguishable without colour | line style or marker |
| Legend, with entries that say what the curve is | not `series 1` |
| Sensible figure size | around 6 by 3.5 inches for one column |
| Saved at 200 dpi or as a vector | not screenshotted |
| A residual panel wherever a model is compared with data | the difference, not the overlay |

## What you have done

You can build a figure with the object-oriented interface and know why it is
preferable; you can draw a labelled family of curves distinguishable in
greyscale; you know when a log axis is compulsory and what a straight line on
one means; you can build the data-and-residual pair that every model comparison
in this course uses; and you can save at a resolution that survives a report.

## Ex_1 is complete

Look back at what the three notebooks were for. Notebook 01 gave you the
language. Notebook 02 gave you the shape rule that everything downstream
depends on. This one gave you the means to show a result.

Ex_2 begins with the same three ideas in PyTorch — tensors instead of arrays,
the same shapes, the same broadcasting rule — and then asks the question the
whole of Part 2 is built on: given a function, get its derivative with respect
to its input, exactly, without ever writing the derivative down.